In [ ]:
!pip install transformers datasets optimum pandas pyarrow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Load and Inspect the Span Detection Data

In [ ]:
import pandas as pd

# Load the span detection training data
df_span = pd.read_parquet("train.parquet")

# Print column names and a few rows to see the structure
print("Columns in span detection data:", df_span.columns.to_list())
print(df_span.head())

Columns in span detection data: ['id', 'content', 'lang', 'manipulative', 'techniques', 'trigger_words']
                                     id  \
0  0bb0c7fa-101b-4583-a5f9-9d503339141c   
1  7159f802-6f99-4e9d-97bd-6f565a4a0fae   
2  e6a427f1-211f-405f-bd8b-70798458d656   
3  1647a352-4cd3-40f6-bfa1-d87d42e34eea   
4  9c01de00-841f-4b50-9407-104e9ffb03bf   

                                             content lang  manipulative  \
0  Новий огляд мапи DeepState від російського вій...   uk          True   
1  Недавно 95 квартал жёстко поглумился над русск...   ru          True   
2  🤩\nТим часом йде евакуація Бєлгородського авто...   uk          True   
3  В Україні найближчим часом мають намір посилит...   uk         False   
4  Расчёты 122-мм САУ 2С1 "Гвоздика" 132-й бригад...   ru          True   

                          techniques  \
0        [euphoria, loaded_language]   
1  [loaded_language, cherry_picking]   
2        [loaded_language, euphoria]   
3                        

In [ ]:
print(df_span[["id", "content", "trigger_words"]].head())

                                     id  \
0  0bb0c7fa-101b-4583-a5f9-9d503339141c   
1  7159f802-6f99-4e9d-97bd-6f565a4a0fae   
2  e6a427f1-211f-405f-bd8b-70798458d656   
3  1647a352-4cd3-40f6-bfa1-d87d42e34eea   
4  9c01de00-841f-4b50-9407-104e9ffb03bf   

                                             content  \
0  Новий огляд мапи DeepState від російського вій...   
1  Недавно 95 квартал жёстко поглумился над русск...   
2  🤩\nТим часом йде евакуація Бєлгородського авто...   
3  В Україні найближчим часом мають намір посилит...   
4  Расчёты 122-мм САУ 2С1 "Гвоздика" 132-й бригад...   

                                   trigger_words  
0    [[27, 63], [65, 88], [90, 183], [186, 308]]  
1  [[0, 40], [123, 137], [180, 251], [253, 274]]  
2                                    [[55, 100]]  
3                                           None  
4                                   [[114, 144]]  


Parse the "trigger_words" Column

In [ ]:
import ast
import numpy as np
import pandas as pd

def parse_trigger_words(row):
    # If row is already a NumPy array, list, or tuple, convert it to a Python list directly.
    if isinstance(row, (np.ndarray, list, tuple)):
        return list(row)

    # Now row should be a scalar (e.g. a string).
    # Check if row is None or NaN
    if row is None or pd.isna(row):
        return []

    # Convert to string and strip whitespace
    row_str = str(row).strip()
    if row_str == "None":
        return []

    try:
        parsed = ast.literal_eval(row_str)
        if isinstance(parsed, list):
            return parsed
        else:
            return []
    except Exception as e:
        print(f"Error parsing trigger_words: {row_str} -> {e}")
        return []

# Now apply the function to the 'trigger_words' column
df_span["trigger_spans"] = df_span["trigger_words"].apply(parse_trigger_words)
print(df_span[["trigger_words", "trigger_spans"]].head())

                                   trigger_words  \
0    [[27, 63], [65, 88], [90, 183], [186, 308]]   
1  [[0, 40], [123, 137], [180, 251], [253, 274]]   
2                                    [[55, 100]]   
3                                           None   
4                                   [[114, 144]]   

                                   trigger_spans  
0    [[27, 63], [65, 88], [90, 183], [186, 308]]  
1  [[0, 40], [123, 137], [180, 251], [253, 274]]  
2                                    [[55, 100]]  
3                                             []  
4                                   [[114, 144]]  


Rename the "content" Column to "text"

In [ ]:
df_span.rename(columns={"content": "text"}, inplace=True)
print(df_span[["text", "trigger_spans"]].head())

                                                text  \
0  Новий огляд мапи DeepState від російського вій...   
1  Недавно 95 квартал жёстко поглумился над русск...   
2  🤩\nТим часом йде евакуація Бєлгородського авто...   
3  В Україні найближчим часом мають намір посилит...   
4  Расчёты 122-мм САУ 2С1 "Гвоздика" 132-й бригад...   

                                   trigger_spans  
0    [[27, 63], [65, 88], [90, 183], [186, 308]]  
1  [[0, 40], [123, 137], [180, 251], [253, 274]]  
2                                    [[55, 100]]  
3                                             []  
4                                   [[114, 144]]  


Convert the DataFrame to a Hugging Face Dataset

In [ ]:
from datasets import Dataset

# Create a subset DataFrame with only the necessary columns
df_subset = df_span[["text", "trigger_spans"]].copy()

# Convert the DataFrame to a Hugging Face Dataset
dataset_span = Dataset.from_pandas(df_subset)
print("Dataset shape:", dataset_span.shape)
print(dataset_span[0])

Dataset shape: (3822, 2)
{'text': 'Новий огляд мапи DeepState від російського військового експерта, кухара путіна 2 розряду, спеціаліста по снарядному голоду та ректора музичної академії міноборони рф Євгєнія Пригожина. \nПригожин прогнозує, що невдовзі настане день звільнення Криму і день розпаду росії. Каже, що передумови цього вже створені. \n*Відео взяли з каналу \nФД\n. \n@informnapalm', 'trigger_spans': [[27, 63], [65, 88], [90, 183], [186, 308]]}


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

# For example, use XLM-RoBERTa, which has a fast tokenizer.
model_path = "benjamin/roberta-large-wechsel-ukrainian"  # or another model known to have a fast tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
model_token = AutoModelForTokenClassification.from_pretrained(
    model_path,
    num_labels=2,
    problem_type="token_classification",
    trust_remote_code=True
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.95M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.hf.co/repos/35/c8/35c82053a323665eb4003e718cf4427ac69e2424fcb33cefb0079d2ac011b94e/af8d06b9df43754bd1d8bf87d05f3f089fc80277275b98fb7c7bb0eb49b3d07e?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1743452342&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MzQ1MjM0Mn19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy8zNS9jOC8zNWM4MjA1M2EzMjM2NjVlYjQwMDNlNzE4Y2Y0NDI3YWM2OWUyNDI0ZmNiMzNjZWZiMDA3OWQyYWMwMTFiOTRlL2FmOGQwNmI5ZGY0Mzc1NGJkMWQ4YmY4N2QwNWYzZjA4OWZjODAyNzcyNzViOThmYjdjN2JiMGViNDliM2QwN2U%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=gd5J0aoMtTfdQIOt6gv22Q50sLfXV8rnZO4myV42LuQmttTWQpGk7jDx%7E4ILcANSXwz-lHBrnVEBO0IIcyl2io5PcO78h0g1Aw-akPhdjno-wRdhexlv4iFE0VI9kRkGoe4QlO5YVophxxp2pl2Q60XdRxMpcryDC6Uvk07DGtUZ51snQEvWGdCJvWrfjrygIRgAeCr-Y7jOL-DpgZj7HcagVbCqZWkecBvuJQUqGDgzsI0qrKEwoyW86us5ym7wedpQUDqt3

model.safetensors:  93%|#########2| 1.32G/1.42G [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at benjamin/roberta-large-wechsel-ukrainian and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenization and Span Alignment Function

In [ ]:
def tokenize_and_align_spans(example):
    # Tokenize the text and return offsets
    tokenized_inputs = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_offsets_mapping=True,
    )

    # Pop the offset mapping from tokenized inputs
    offset_mapping = tokenized_inputs.pop("offset_mapping")

    # Initialize token-level labels: 0 for non-manipulative
    labels = [0] * len(offset_mapping)

    # Get the trigger spans for this example (each span is a list, e.g., [start, end])
    spans = example["trigger_spans"]

    # For each token, check if its character offsets overlap with any trigger span.
    for i, (token_start, token_end) in enumerate(offset_mapping):
        # For special tokens and padding, offset mapping is (0, 0) → label as -100 (ignored in loss)
        if token_start == token_end:
            labels[i] = -100
            continue

        # Check each span for overlap with the token span
        for span in spans:
            span_start, span_end = span
            # Overlap condition: token_end > span_start and token_start < span_end
            if token_end > span_start and token_start < span_end:
                labels[i] = 1
                break  # Once overlapping with one span, no need to check further

    # Add the labels to tokenized inputs
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply the function to the dataset (process one example at a time)
dataset_span = dataset_span.map(tokenize_and_align_spans, batched=False)

# Optionally, inspect one example to verify
print(dataset_span[0])

Map:   0%|          | 0/3822 [00:00<?, ? examples/s]

{'text': 'Новий огляд мапи DeepState від російського військового експерта, кухара путіна 2 розряду, спеціаліста по снарядному голоду та ректора музичної академії міноборони рф Євгєнія Пригожина. \nПригожин прогнозує, що невдовзі настане день звільнення Криму і день розпаду росії. Каже, що передумови цього вже створені. \n*Відео взяли з каналу \nФД\n. \n@informnapalm', 'trigger_spans': [[27, 63], [65, 88], [90, 183], [186, 308]], 'input_ids': [0, 14111, 8398, 454, 773, 16715, 6338, 13664, 9174, 343, 7955, 8909, 23596, 16, 6560, 613, 2865, 9045, 406, 25421, 16, 14479, 358, 47969, 612, 15393, 330, 22592, 18815, 6629, 4590, 16197, 546, 328, 2894, 295, 302, 4880, 1035, 295, 1920, 313, 18, 211, 190, 1790, 295, 1920, 250, 44352, 16, 383, 21151, 34882, 1573, 6064, 4874, 306, 1573, 25956, 751, 1291, 18, 36796, 16, 383, 24125, 920, 962, 11252, 18, 211, 190, 14, 29653, 4693, 270, 10775, 211, 190, 934, 545, 190, 18, 211, 190, 36, 1017, 14526, 82, 4209, 1489, 81, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

Convert the Dataset to PyTorch Format and Split It

In [ ]:
# Set the dataset format to PyTorch (if not done already)
dataset_span.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# Split the dataset into train and evaluation sets (e.g., 80/20 split)
split_dataset_span = dataset_span.train_test_split(test_size=0.2, seed=42)
train_dataset_span = split_dataset_span["train"]
eval_dataset_span = split_dataset_span["test"]

print("Training examples:", len(train_dataset_span))
print("Evaluation examples:", len(eval_dataset_span))

Training examples: 3057
Evaluation examples: 765


Define a Compute Metrics Function

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def compute_token_metrics(eval_pred):
    logits, labels = eval_pred
    # Use argmax over the last dimension to get a single class (0 or 1) per token
    preds = np.argmax(logits, axis=-1)

    preds_flat = []
    labels_flat = []
    for p, l in zip(preds, labels):
        for pred, lab in zip(p, l):
            if lab != -100:
                preds_flat.append(pred)
                labels_flat.append(lab)

    f1 = f1_score(labels_flat, preds_flat, average="macro", zero_division=0)
    return {"token_macro_f1": f1}

Define Training Arguments for the Token Classification Task

In [ ]:
from transformers import TrainingArguments

training_args_token = TrainingArguments(
    output_dir="./results-span",
    evaluation_strategy="steps",
    eval_steps=100,             # Evaluate every 100 steps
    save_steps=100,             # Save a checkpoint every 100 steps
    num_train_epochs=8,         # Number of epochs (adjust based on dataset size)
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    fp16=True,                  # Mixed precision for T4 GPU
    logging_steps=50,
    save_total_limit=2,         # Keep only the two most recent checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="token_macro_f1",
    greater_is_better=True,
    report_to="none"            # Disable wandb logging if desired
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Initialize the Trainer and Fine-Tune the Model

In [ ]:
from transformers import Trainer

trainer_token = Trainer(
    model=model_token,
    args=training_args_token,
    train_dataset=train_dataset_span,
    eval_dataset=eval_dataset_span,
    compute_metrics=compute_token_metrics
)

trainer_token.train()

Step,Training Loss,Validation Loss,Token Macro F1
100,0.427300,0.398898,0.723316
200,0.387700,0.423702,0.700930
300,0.340600,0.434920,0.732851
400,0.284700,0.447674,0.744218
500,0.242200,0.467350,0.747224
600,0.213500,0.489536,0.734043
700,0.198700,0.516390,0.735017


TrainOutput(global_step=768, training_loss=0.30392689754565555, metrics={'train_runtime': 1186.6062, 'train_samples_per_second': 20.61, 'train_steps_per_second': 0.647, 'total_flos': 5678114549870592.0, 'train_loss': 0.30392689754565555, 'epoch': 8.0})

Evaluate and Save the Fine-Tuned Model

In [ ]:
eval_results_token = trainer_token.evaluate()
print("Token classification evaluation results:", eval_results_token)

# Save the model and tokenizer
trainer_token.save_model("./roberta-large-ukrainian_2")
tokenizer.save_pretrained("./roberta-large-ukrainian_2")

Token classification evaluation results: {'eval_loss': 0.4673503339290619, 'eval_token_macro_f1': 0.7472239405471415, 'eval_runtime': 4.0493, 'eval_samples_per_second': 188.919, 'eval_steps_per_second': 5.927, 'epoch': 8.0}


('./roberta-large-ukrainian_2/tokenizer_config.json',
 './roberta-large-ukrainian_2/special_tokens_map.json',
 './roberta-large-ukrainian_2/vocab.json',
 './roberta-large-ukrainian_2/merges.txt',
 './roberta-large-ukrainian_2/added_tokens.json',
 './roberta-large-ukrainian_2/tokenizer.json')

Submission and test data predictions

In [ ]:
import pandas as pd

# Load the test set (ensure the file is in your working directory)
df_test = pd.read_csv("test.csv")

# Inspect the first few rows
print(df_test.head())

                                     id  \
0  521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba   
1  9b2a61e4-d14e-4ff7-b304-e73d720319bf   
2  f0f1c236-80a8-4d25-b30c-a420a39be632   
3  31ea05ba-2c2b-4b84-aba7-f3cf6841b204   
4  a79e13ec-6d9a-40b5-b54c-7f4f743a7525   

                                             content  
0  Они просрали нашу технику, положили кучу людей...  
1  ❗️\nКитай предлагает отдать оккупированные тер...  
2  Сегодня будет ровно 6 месяцев с этого обещания...  
3  ⚡️\nІзраїль вперше у світі збив балістичну рак...  
4  Склав невелику навчально-методичну таблицю на ...  


In [ ]:
df_test.rename(columns={"content": "text"}, inplace=True)

In [ ]:
def predict_spans(text):
    # Tokenize the input text and return offsets as tensors
    tokenized_inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    # Extract offset mapping and convert it to a list (on CPU)
    offset_mapping = tokenized_inputs.pop("offset_mapping")  # this remains on CPU
    offset_mapping = offset_mapping[0].tolist()

    # Move the rest of the inputs to the same device as the model
    device = next(model_token.parameters()).device
    tokenized_inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}

    # Forward pass
    with torch.no_grad():
        outputs = model_token(**tokenized_inputs)
    logits = outputs.logits  # shape: (1, seq_length, 2)

    # Get predictions (choose the class with the highest logit for each token)
    preds = torch.argmax(logits, dim=-1).squeeze().cpu().numpy()

    # Convert token-level predictions to character-level spans
    spans = []
    current_span = None
    for (token_start, token_end), pred in zip(offset_mapping, preds):
        if token_start == token_end:
            # Skip special tokens or padding
            continue
        if pred == 1:
            # If not currently in a span, start one; else extend current span
            if current_span is None:
                current_span = [token_start, token_end]
            else:
                current_span[1] = token_end
        else:
            if current_span is not None:
                spans.append(tuple(current_span))
                current_span = None
    if current_span is not None:
        spans.append(tuple(current_span))
    return spans

# Test on a sample text
sample_text = "Пакують українців у мінівени!"
print("Predicted spans for sample:", predict_spans(sample_text))

NameError: name 'torch' is not defined

In [ ]:
submission_rows = []

for idx, row in df_test.iterrows():
    example_id = row["id"]
    text = row["text"]
    spans = predict_spans(text)
    # Convert the list of spans to a string representation
    spans_str = str(spans)
    submission_rows.append({"id": example_id, "trigger_words": spans_str})

# Create a DataFrame for submission
submission_df = pd.DataFrame(submission_rows)

# Inspect the submission DataFrame
print(submission_df.head())

# Save the submission file as CSV (ensure to include header and no index)
submission_df.to_csv("submission.csv", index=False)